## WCGAN

### Wasserstein Conditional Generative Adversarial Network

From scratch training of the WC-GAN model modified for Audio to Image scene generation

Author: Alexi

In [1]:
from wcgan import Generator, Discriminator
from trainer import train
from dataloader import *
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

TEST_SIZE = 0.2
CSV_PATH = "main_dataV3.csv"

SEED = 42

AUDIO_ENCODER = "laion/clap-htsat-unfused"
noise_dim = 128
audio_embed_dim = 512  # must match your audio embedding model's dims


c:\Users\alexi\anaconda3\envs\audio_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

datasetPath = r"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound"

df = pd.read_csv(CSV_PATH)
df['base_folder'] = df['base_folder'].apply(lambda x: os.path.join(datasetPath, x))

def check_files_exist(row):
    base_folder = row['base_folder']
    image_path = os.path.join(base_folder, "image", row['image_file'])
    audio_path = os.path.join(base_folder, "audio", row['audio_file'])
    return os.path.exists(image_path) and os.path.exists(audio_path)

print(f"Initial dataset size: {len(df)}")

# Filter dataframe to only rows where both files exist
df = df[df.apply(check_files_exist, axis=1)].reset_index(drop=True)

print(f"Filtered dataset size (files exist): {len(df)}")

df.head()




Initial dataset size: 2575
Filtered dataset size (files exist): 2575


,base_folder,image_file,audio_file,caption
0,D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggs...,Bu2xe4OSo_430.png,Bu2xe4OSo_000430.wav,wind noise
1,D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggs...,CudrykwoE_67.png,CudrykwoE_000067.wav,cricket chirping
2,D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggs...,DTR0_mIGI_11.png,DTR0_mIGI_000011.wav,people battle cry
3,D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggs...,Lj4Y_96f0_120.png,Lj4Y_96f0_000120.wav,"bee, wasp, etc. buzzing"
4,D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggs...,PlJNEnf-s_288.png,PlJNEnf-s_000288.wav,"bee, wasp, etc. buzzing"


In [3]:
# Train-test split (e.g., 80% train, 20% test)
print("Splitting dataset into train and test sets...")
# train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED)

# Step 1: Precompute audio embeddings and build FAISS index on the training set
# faiss_index_train, embeddings_array_train = build_faiss_index(train_df, clap_model_name=AUDIO_ENCODER, device='cuda', save_dir="trainData", index_filename="faiss_index_train.bin", embeddings_filename="embeddings_train.npy", csv_filename = "metadata_train.csv")
load_dir = "trainData"
faiss_index_train, embeddings_array_train, train_df = load_faiss_index_embeddings_metadata(load_dir, index_filename="faiss_index_train.bin", embeddings_filename="embeddings_train.npy", df_filename="metadata_train.csv")

print("FAISS index and embeddings for training set ready.")
# Step 2: Create train dataloader using training split and embeddings
train_loader = get_dataloader_from_faiss(train_df, faiss_index_train, embeddings_array_train, batch_size=16, img_size=(64, 64), base_dir=datasetPath)

print("Train dataloader ready.")
# For test set, optionally precompute embeddings and build FAISS index separately
# faiss_index_test, embeddings_array_test = build_faiss_index(test_df, clap_model_name=AUDIO_ENCODER, device='cuda', save_dir="testData", index_filename="faiss_index_test.bin", embeddings_filename="embeddings_test.npy", csv_filename = "metadata_test.csv")
load_dir = "testData"
faiss_index_test, embeddings_array_test, test_df = load_faiss_index_embeddings_metadata(load_dir, index_filename="faiss_index_test.bin", embeddings_filename="embeddings_test.npy", df_filename="metadata_test.csv")
print("FAISS index and embeddings for test set ready.")

test_loader = get_dataloader_from_faiss(test_df, faiss_index_test, embeddings_array_test, batch_size=16, img_size=(64, 64), base_dir=datasetPath)
print("Test dataloader ready.")


print(f"Training samples: {len(train_df)}, Testing samples: {len(test_df)}")


Splitting dataset into train and test sets...
Loaded FAISS index from trainData\faiss_index_train.bin
Loaded embeddings array from trainData\embeddings_train.npy with shape (2060, 512)
Loaded dataset metadata from trainData\metadata_train.csv with 2060 rows
FAISS index and embeddings for training set ready.
Train dataloader ready.
Loaded FAISS index from testData\faiss_index_test.bin
Loaded embeddings array from testData\embeddings_test.npy with shape (515, 512)
Loaded dataset metadata from testData\metadata_test.csv with 515 rows
FAISS index and embeddings for test set ready.
Test dataloader ready.
Training samples: 2060, Testing samples: 515


In [4]:
import numpy as np

def check_faiss_pack(name, faiss_index, embeds, df):
    print(f"\n== {name} integrity check ==")
    # 1) sizes match?
    n_index = faiss_index.ntotal
    n_emb   = embeds.shape[0]
    n_df    = len(df)
    print(f"Index: {n_index}, Embeds: {n_emb}, Rows: {n_df}")
    assert n_index == n_emb == n_df, f"Count mismatch: {n_index} vs {n_emb} vs {n_df}"

    # 2) embedding norms sane?
    norms = np.linalg.norm(embeds, axis=1)
    print("Embedding norms: mean", norms.mean(), "std", norms.std(), "min", norms.min(), "max", norms.max())
    assert np.isfinite(norms).all(), "Found non-finite norms"
    assert norms.mean() > 0.1, "Embeddings look near-zero; encoder/normalization issue?"

    # 3) self-nearest neighbor should be itself (distance≈0)
    import faiss
    D, I = faiss_index.search(embeds[:8], 1)
    mism = (I[:,0] != np.arange(8)).sum()
    print(f"Self-NN mismatches in first 8: {mism} (should be 0); D (avg)={D.mean():.4f}")




In [5]:
check_faiss_pack("TRAIN", faiss_index_train, embeddings_array_train, train_df)
check_faiss_pack("TEST",  faiss_index_test,  embeddings_array_test,  test_df)




== TRAIN integrity check ==
Index: 2060, Embeds: 2060, Rows: 2060
Embedding norms: mean 1.0 std 3.908979e-08 min 0.9999999 max 1.0000001
Self-NN mismatches in first 8: 0 (should be 0); D (avg)=0.0000

== TEST integrity check ==
Index: 515, Embeds: 515, Rows: 515
Embedding norms: mean 1.0 std 3.6203726e-08 min 0.9999999 max 1.0000001
Self-NN mismatches in first 8: 0 (should be 0); D (avg)=0.0000


In [6]:
def load_checkpoint(ckpt_path, generator, discriminator, g_opt=None, d_opt=None, ema_G=None, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device)

    generator.load_state_dict(ckpt["generator_state_dict"], strict=False)
    discriminator.load_state_dict(ckpt["discriminator_state_dict"], strict=False)

    if g_opt is not None and "g_optimizer_state_dict" in ckpt:
        g_opt.load_state_dict(ckpt["g_optimizer_state_dict"])
    if d_opt is not None and "d_optimizer_state_dict" in ckpt:
        d_opt.load_state_dict(ckpt["d_optimizer_state_dict"])

    if ema_G is not None and "ema_G_state_dict" in ckpt:
        ema_G.load_state_dict(ckpt["ema_G_state_dict"], strict=False)

    start_epoch = ckpt.get("epoch", -1) + 1
    global_step = ckpt.get("global_step", 0)
    return start_epoch, global_step


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from wcgan import build_models

# 64×64
generator, discriminator = build_models(resolution=64, 
                                        noise_dim=128, 
                                        audio_embed_dim=512, 
                                        base_channels=64
                                        )


In [8]:
from copy import deepcopy

# optimizers
g_opt = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.0, 0.9))
d_opt = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.0, 0.9))

# EMA copy (same as in your trainer)
ema_G = deepcopy(generator).eval()
for p in ema_G.parameters(): p.requires_grad_(False)

ckpt_path = fr"models\ckpt_epoch90.pth"
start_epoch, global_step = load_checkpoint(ckpt_path, generator, discriminator, g_opt, d_opt, ema_G, device=device)
print(f"Resuming from epoch {start_epoch}, global_step {global_step}")


C:\Users\alexi\AppData\Local\Temp\ipykernel_46492\3095504127.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


Resuming from epoch 91, global_step 11739



Modifications:

+ConditionalBatchNorm2d

+Projection D gives a stronger, smoother conditional signal than concatenating at the last linear layer, and removing BN from D stabilizes WGAN-GP.

+BatchNorm in the Discriminator plus spectral_norm and GP. That combo is notoriously unstable.

-ConvTranspose2d layers replaced with Upsample + Conv2d layers for better stability


In [10]:
train(
    dataloader=train_loader,
    generator=generator,
    discriminator=discriminator,
    device=device,
    noise_dim=noise_dim,
    audio_embed_dim=audio_embed_dim,
    n_epochs=200,
    g_lr=2e-4,
    d_lr=1e-4,
    lambda_gp=15.0,        # stronger gradient penalty for stability
    n_critic=4,            # fewer D updates per G
    lambda_fm=10.0,        # feature matching loss weight
    use_lpips=True,        # perceptual loss
    lambda_lpips=0.2,      # LPIPS weight
    ema_decay=0.999,       # EMA smoothing factor
    ema_start_step=1000,   # start EMA after 1000 steps
    print_every=100,       # progress print interval
    plot_every=200,        # update loss curve every 200 steps
    inference_every=500,   # save generated samples every 500 steps
    save_path="models",    # checkpoint folder
    output_folder="outputs",
    loss_plot_path="loss_plot.png"
)


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\alexi\anaconda3\envs\audio_env\lib\site-packages\lpips\weights\v0.1\vgg.pth


Epoch 200/200: 100%|██████████| 129/129 [00:46<00:00,  2.76it/s, D_fake=419, D_loss=-2.47, D_real=421, FM=0.383, GP=0.00794, G_loss=-415, LPIPS=0.607]


### 1. Generator (G) Architectural Updates

The generator incorporates several state-of-the-art techniques to improve visual quality and fidelity to the audio input.

#### SPADE Residual Blocks

The core generative block has been upgraded to SPADE ResBlocks (Spatially Adaptive Denormalization).

Structure: The block now utilizes two SPADE norms per residual block:

- norm1 is applied before the first ReLU and convolution (norm1 → ReLU → (upsample) → conv1).

- norm2 is applied before the second ReLU and convolution (norm2 → ReLU → conv2).

Function: This approach allows the semantic (or, in this case, audio-conditioned) layout to more directly influence the intermediate feature maps, which is critical for generating high-quality, conditioned output.

Skip Path: Includes an upsampled path and a $1 \times 1$ convolutional projection when resolution changes, ensuring effective information flow.

#### Self-Attention Modules

Self-Attention is used to model long-range dependencies across the spatial dimensions of the feature maps, improving global coherence.

- Placement: Self-Attention modules are now inserted at configurable spatial sizes, specifically at 32x32 and 64x64 resolutions (attn_at=(32, 64)).

- Implementation: The attention layers are managed via a ModuleDict, with a fixed lookup mechanism (if key in self.attn_layers) ensuring they are only activated at the specified feature map resolutions.

#### Audio Conditioning (MLP and Normalization)

The audio input is now processed to create a robust and normalized conditioning signal.

- Conditioning MLP: A small Multilayer Perceptron (Linear → ReLU → Linear) processes the raw audio embedding.

- Normalization: The final output of the MLP is explicitly normalized using F.normalize(..., dim=1) before being used in the SPADE layers and concatenated elsewhere, ensuring stable, magnitude-independent conditioning.

#### Configuration and Output

The generator is now highly flexible in terms of output resolution and capacity.

- Configurable Resolution (num_upsamples): The final output resolution (e.g., 64, 128, or 256 pixels) is determined by the number of upsample blocks used, starting from an initial $4 \times 4$ feature map with base_channels * 8.

- Configurable Capacity (base_channels): The width and depth of the network are controlled by the base_channels parameter (e.g., 64, 96, 128), allowing for easy scaling of model complexity.

- to_rgb Head: The final output layer structure is standardized for quality: BN → ReLU → 3x3 conv → Tanh.

### 2. Discriminator (D) Architectural Updates

The discriminator switches from a simple concatenated MLP to a more powerful Projection Discriminator, which is specifically designed for class (or, in this case, conditional) guided generation.

#### Projection Discriminator Backbone

The architecture is designed for robust feature extraction without traditional normalization.

- Type: The Projection Discriminator replaces the former "final concat" MLP approach for conditioning.

- Layers: The backbone uses standard architecture components: strided convolutions with Spectral Normalization (SN) applied to weights, and LeakyReLU activations, with a deliberate absence of BatchNorm layers.

#### Realness and Conditioning Heads

The final scoring mechanism separates realness and conditioning into two paths which interact via a dot product.

- Pooled Features (h): Global Average Pooling (GAP) is applied to the final feature maps to produce a pooled feature vector h.

- Realness Head: The primary realness score is derived from these features: fc_real(h).

- Conditioning Head (e): The normalized audio embedding (e) is projected into a weight vector w using fc_proj(e).

- Final Score: The conditioning influence is calculated as a dot product of the pooled features and the projected audio weights (h · w). This result is then added to the primary realness score, yielding the final discrimination value.

####  Audio Normalization and Feature Extraction

- Audio Normalization: Similar to the generator, the audio embedding e is explicitly normalized (e = F.normalize(audio_embedding, dim=1)) before being used in the projection head, ensuring stable conditional inputs.

- features() Method: A new method is added to return the pooled penultimate features [B, C] (the feature vector h). This is critical for the new Feature Matching loss implemented during training.

#### 3. Training and Loss Enhancements

These additions focus on improving training stability and pushing the generated output towards higher perceptual fidelity.

#### EMA Generator for Inference

A copy of the generator's weights is maintained using Exponential Moving Average (EMA).

- Function: The EMA model (or "shadow model") is updated slowly based on the primary generator's weights.

- Usage: It is used exclusively for inference, providing significantly more stable and higher-quality outputs compared to using the last-trained weights of the primary generator.

#### Feature Matching Loss

This loss function stabilizes training and encourages the generator to produce samples that look "real" in the feature space.

- Mechanism: It utilizes the new D.features() method to compute the distance (typically L1 or L2) between the pooled feature vectors h generated by the fake sample and those generated by real samples.

- Goal: This forces the generator to match the structural characteristics captured by the discriminator's internal representations.

#### LPIPS Perceptual Loss

LPIPS (Learned Perceptual Image Patch Similarity) is introduced as a measure of visual quality.

- Mechanism: LPIPS uses a pre-trained deep neural network (e.g., VGG) to measure the perceptual difference between the generated image and the target image.

- Goal: Unlike simple pixel-wise losses (L1/L2), LPIPS aligns much better with human judgment, encouraging the generated images to be perceptually sharp and realistic.

These modifications represent a comprehensive upgrade, moving the system towards a high-performance, conditioned GAN setup capable of flexible resolution and improved visual quality.